# KAT-VAD v2 — **Nexar CRN diagnostic (Amendment 19, §30)**: is the CRN arms' crop cost the reference unit?

Rules `core/docs/v2/PREREG_ADDENDUM.md` **§30** (C1–C9). A diagnostic: it decides no arm and never moves `B`.
**`nexar_val` only** — `nexar_test` was read once by §29 and stays closed (D-N22). No training.

Training baked each window's CRN reference (R2) over its **source video** (~40 s, event ≈ 2–5 %); §29's eval
re-referenced each 8 s **crop** by itself (event ≈ 35 %). This notebook re-scores the Nexar-trained CRN arms
(A3 primary; A1, A3-ign printed) on `nexar_val` with `nexar_eval --crn-reference source` — every input row is the
same, only the reference is the whole video's — and compares it to §29's `val/<arm>/` reads.

| step | what | GPU? |
|---|---|---|
| 0 | setup, deps | — |
| 1 | preflight: §30 signed, §29's val reads present, J10 on the 15 CRN checkpoints | no |
| 2 | score A1 / A3 / A3-ign on `nexar_val` with `--crn-reference source` → `val_crnref/<arm>/` | **yes** (~minutes) |
| 3 | `crn_cost_diag ref` (C4 gate → C5/C6 sentence) + `share` (C7) → paste **`crn_ref.md`** to Claude | no |

Every step skips what is already on Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read: data/, cache/
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read: kat-vad/ (code), cache/, outputs/; write: outputs/
os.environ['PROJECT_ROOT']       = DRIVE_V2
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_V2}/outputs'

os.environ['REPO'] = f'{DRIVE_V2}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v2 there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert hasattr(constants, 'CRN_REFERENCES'), 'this upload predates Amendment 19 (§30) -- re-upload branch v2'

DATASET  = constants.NEXAR_DATASET
CRN      = 'R2'
ENC      = constants.VIDEOMAE_ENCODER_S                  # V2-S
OFF      = constants.V2_OFF
SEEDS    = list(constants.NEXAR_RUN_SEEDS)               # 2024-2028 (§29 D-N17)
C        = constants.NEXAR_CONSTRUCTION_WINDOW
EXPECT   = {'A0': (OFF, OFF), 'A1': (CRN, OFF), 'A2': (OFF, ENC), 'A3': (CRN, ENC), 'A3ign': (CRN, ENC)}
BAKE_TAG = {'A1': f'A1_{CRN}', 'A2': 'A2_S', 'A3': f'A3_{CRN}_S'}
BAKE_OF  = {'A1': 'A1', 'A2': 'A2', 'A3': 'A3', 'A3ign': 'A3'}
CRN_ARMS = sorted({constants.CRN_DIAG_PRIMARY[0], *(a for a, _ in constants.CRN_DIAG_PRINTED)})  # A1, A3, A3ign
ALL_READ = sorted({a for pair in (constants.CRN_DIAG_PRIMARY, *constants.CRN_DIAG_PRINTED) for a in pair})

CENSUS   = constants.OUTPUT_ROOT / 'v2' / 'REPORTS' / 'nexar_n0' / 'census.json'
V2_CACHE = Path(DRIVE_V2) / 'cache'
CLIP_S1  = V2_CACHE / 'clip' / constants.NEXAR_CLIP_S1_DATASET
VIDEO_S1 = V2_CACHE / 'video' / ENC / f'{constants.NEXAR_VIDEO_S1_DATASET}_squash'
BAKES    = {b: V2_CACHE / 'v2' / tag / 'Nexar_s8' for b, tag in BAKE_TAG.items()}
DATA     = Path(DRIVE_V2) / 'data' / f'{DATASET}_{C}'
BUILD    = constants.OUTPUT_ROOT / 'v2' / 'REPORTS' / 'nexar_n4' / 'build_report.json'
RUNS     = constants.OUTPUT_ROOT / constants.NEXAR_FULL_RUNS_DIR   # v2_nexar_window/
VAL_ROOT = RUNS / 'val'                                            # §29's item-reference reads
SRC_ROOT = RUNS / 'val_crnref'                                     # this notebook's source-reference reads
DIAG     = RUNS / 'REPORTS' / 'crn_diag'
LOCAL    = Path('/content/nexar_crnref')
LOCAL.mkdir(parents=True, exist_ok=True)
for name in ('CENSUS', 'CLIP_S1', 'VIDEO_S1', 'DATA', 'RUNS', 'SRC_ROOT'):
    print(f'{name:9s}', globals()[name])
print('source-reference arms:', CRN_ARMS, '| item reads needed:', ALL_READ)

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu
python -c "import transformers, av, einops, faiss; print('transformers', transformers.__version__)"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- step 2 will be slow'

In [ ]:
import json
import pickle
import shutil
import subprocess
import time

import numpy as np
import torch
import yaml

ENV = dict(os.environ)


def run(cmd, log=None, quiet_steps=False):
    '''Stream a child's output here (and to ``log``); raise with its tail on failure.'''
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                if not (quiet_steps and 'global_step' in line):
                    print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=None):
    '''Copy a Drive dir (or only ``ids`` + its manifests) / file to VM disk once (bit-identical; C2).'''
    if dst.exists() and ids is None:
        return dst
    t0 = time.time()
    if src.is_dir() and ids is None:
        shutil.copytree(src, dst)
    elif src.is_dir():
        dst.mkdir(parents=True, exist_ok=True)
        for v in ids:
            if not (dst / f'{v}.npy').exists():
                shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
        for m in src.glob('*.json'):
            shutil.copy2(m, dst / m.name)
        for m in src.glob('*.npz'):
            shutil.copy2(m, dst / m.name)
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    '''Copy a VM-local result to Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10, lesson (bi)).'''
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def ckpt_step(path):
    '''The checkpoint's own global_step; 0 if absent, -1 if unreadable (never equals a real step).'''
    if not path.exists():
        return 0
    try:
        return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))
    except (RuntimeError, EOFError, pickle.UnpicklingError) as exc:
        print(f'UNREADABLE {path}: {exc}')
        return -1


def metrics_last(path):
    lines = [l for l in path.read_text().splitlines() if l.strip()] if path.exists() else []
    return json.loads(lines[-1]) if lines else None


def write_ids(path, ids):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(''.join(f'{v}\n' for v in ids))
    return path

## 1. Preflight (C2, C4(a))

§30 must be signed in the upload; §29's `val/<arm>/` reads (the `item` path) must exist for A0–A3-ign; every CRN
checkpoint must be finished at the `window` plan's step (J10), and §29's val read must name the same checkpoints.

In [ ]:
from core.data.v2_splits import load_split

addendum = (REPO / 'core/docs/v2/PREREG_ADDENDUM.md').read_text(encoding='utf-8')
assert '## 30. Amendment 19' in addendum, 'upload predates addendum §30 -- re-upload branch v2'
assert 'Authorized by the user on' in addendum.split('## 30. Amendment 19', 1)[1], '§30 is not signed'
assert CENSUS.is_file(), f'missing {CENSUS}'
STEP = json.loads(BUILD.read_text())['plan'][C]['steps']
VAL = load_split(constants.V2_SPLIT_NEXAR_VAL)


def run_dir(arm, seed):
    return RUNS / arm / f's{seed}'


bad = []
for arm in ALL_READ:
    if not (VAL_ROOT / arm / 'nexar_readout.json').is_file():
        bad.append(f'{arm}: no §29 val read at {VAL_ROOT / arm}')
for arm in CRN_ARMS:
    item = json.loads((VAL_ROOT / arm / 'nexar_readout.json').read_text())
    for s in SEEDS:
        d = run_dir(arm, s)
        step, last = ckpt_step(d / 'checkpoint_last.pt'), metrics_last(d / 'metrics.jsonl')
        if not (last is not None and step == last['global_step'] == STEP):
            bad.append(f'{arm} s{s}: J10 (ckpt {step}, metrics {last and last["global_step"]}, want {STEP})')
        if item['checkpoints'].get(f's{s}') != str(d / 'checkpoint_last.pt'):
            bad.append(f'{arm} s{s}: §29 val read scored {item["checkpoints"].get(f"s{s}")}')
assert not bad, 'preflight failed -- paste to Claude:\n' + '\n'.join(bad)
print(f'{len(CRN_ARMS) * len(SEEDS)} CRN checkpoints at step {STEP}; §29 val reads present for {ALL_READ}; '
      f'nexar_val {len(VAL)} videos')

## 2. Score the CRN arms with the source-video reference (GPU)

Same call as `nexar_train.ipynb` step 5 plus `--crn-reference source`; same checkpoint paths, same statistics
(`fit-ids` bake, Nexar train). An arm already read is reused (§30: nothing is re-scored after a number is seen).

In [ ]:
L_VAL_X = stage(CLIP_S1, LOCAL / 'stage' / 'clip_s1_val', VAL)
L_VAL_U = stage(VIDEO_S1, LOCAL / 'stage' / 'video_s1_val', VAL)
for arm in CRN_ARMS:
    out = SRC_ROOT / arm
    if (out / 'nexar_readout.json').exists():
        print(f'{arm}: reusing {out}')
        continue
    runs = [x for s in SEEDS for x in ('--run', f's{s}', run_dir(arm, s) / 'checkpoint_last.pt')]
    video = ['--video-s1-dir', L_VAL_U] if EXPECT[arm][1] != OFF else []
    local = LOCAL / 'val_crnref' / arm
    run([sys.executable, '-m', 'core.tools.nexar_eval', *runs, '--census', CENSUS, '--clip-s1-dir', L_VAL_X,
         '--stats-dir', BAKES[BAKE_OF[arm]], *video, '--data-dir', DATA,
         '--split', constants.V2_SPLIT_NEXAR_VAL, '--crn-reference', constants.CRN_REFERENCE_SOURCE,
         '--out-dir', local], log=LOCAL / f'{arm}_crnref.log')
    publish(local, out)
    print(f'{arm}: -> {out}')

## 3. Read-out: C4 gate → C6 sentence (`crn_ref.md`)

The gate stops before any number if a whole-video item differs between the two paths by more than
`CRN_DIAG_PARITY_TOL`. **Paste `crn_ref.md` to Claude.**

C7's share table (`crn_cost_diag share`) reads ~3,000 per-video score files and is run locally on the synced outputs (`RESULTS_NEXAR.md` §5.1), not here: over Drive FUSE it takes tens of minutes.


In [ ]:
run([sys.executable, '-m', 'core.tools.crn_cost_diag', 'ref', '--item-dir', VAL_ROOT, '--source-dir', SRC_ROOT,
     '--out-dir', LOCAL / 'diag'])
publish(LOCAL / 'diag', DIAG)
print((DIAG / 'crn_ref.md').read_text())
drive.flush_and_unmount()
print('Drive flushed')
